In [1]:
import os

print(os.getcwd())

C:\Users\LENOVO


In [2]:
import os

folder = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"

print("Files found:")
for file in os.listdir(folder):
    print(file)

Files found:
Hexagon-Dataset - Geography and tourism.csv


In [3]:
import pandas as pd

file_path = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\Hexagon-Dataset - Geography and tourism.csv"

df = pd.read_csv(file_path)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 234
Columns:
['Question ID', 'Domain', 'Subdomain', 'Question Type', 'Difficulty', 'English Question', 'Nepali Question', 'Gold Answer (EN)', 'Gold Answer (NE)', 'Explanation', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'created by', 'Reviewed By', 'Review Status', 'Notes', 'Which LLM?']


In [4]:
df.head()

,Question ID,Domain,Subdomain,Question Type,Difficulty,English Question,Nepali Question,Gold Answer (EN),Gold Answer (NE),Explanation,Source ID,Source Title,Source URL,Source Type,created by,Reviewed By,Review Status,Notes,Which LLM?
0,GEO001,Geography,Physical Geography,Fact Retrieval,Easy,What is the highest mountain in the world?,विश्वको सबैभन्दा अग्लो हिमाल कुन हो?,Mount Everest (Sagarmatha),सगरमाथा (माउन्ट एभरेष्ट),"Mount Everest, on the Nepal-China border, is t...",SD001,Survey Department of Nepal – Peak Heights,https://dos.gov.np/,Government,om,NaN,done,verified,NaN
1,GEO002,Geography,Physical Geography,Fact Retrieval,Medium,"How many mountains above 8,000 meters are loca...","नेपालमा ८,००० मिटरभन्दा माथिका कति हिमालहरू छन्?",8,८,Nepal contains eight of the world's fourteen p...,SD002,Survey Department of Nepal – Eight-Thousanders,https://dos.gov.np/,Government,om,NaN,done,verified,NaN
2,GEO003,Geography,Administrative Geography,Fact Retrieval,Easy,How many provinces does Nepal have?,नेपालमा कति प्रदेश छन्?,7,७,Nepal was restructured into 7 provinces under ...,LAW001,Constitution of Nepal - Nepal Law Commission,https://lawcommission.gov.np/content/13437/nep...,Government,om,NaN,done,verified,NaN
3,GEO004,Geography,Hydrology,Fact Retrieval,Medium,Which river gorge in Nepal is considered one o...,नेपालको कुन नदी उपत्यका विश्वकै गहिरो उपत्यकाम...,Kali Gandaki Gorge,कालीगण्डकी उपत्यका,"The Kali Gandaki Gorge, between the Dhaulagiri...",TB001,Nepal Tourism Board – Trekking Regions,https://ntb.gov.np/,Government,om,NaN,done,verified,NaN
4,GEO005,Geography,Administrative Geography,Fact Retrieval,Hard,Which province was formerly known as Province ...,पहिले 'प्रदेश नं. १' भनिने प्रदेशलाई पछि आधिका...,Koshi Province,कोशी प्रदेश,Province No. 1 was officially renamed Koshi Pr...,KP001,Kathmandu Post - Province 1 named Koshi (1 Mar...,https://kathmandupost.com/province-no-1/2023/0...,News/Government Record,om,NaN,done,verified,NaN


In [6]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print("Status:", response.status_code)

for model in response.json()["models"]:
    print("-", model["name"])

Status: 200
- gemma3:1b


In [7]:
import requests

url = "http://localhost:11434/api/generate"

data = {
    "model": "gemma3:1b",
    "prompt": "What is the highest mountain in the world?",
    "stream": False
}

response = requests.post(url, json=data)

print(response.status_code)
print(response.json()["response"])

200
As of December 29, 2023, the title of the highest point on Earth belongs to **Mount Kilimanjaro**, located in Tanzania.

It stands at a towering height of approximately **5,895 meters (19,340 feet)** above sea level and is an active volcano. 


While Mount Everest occupies a prominent position globally, Kilimanjaro consistently holds the top spot in terms of summit height.




In [8]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

# Total characters in the dataset
total_chars = df.to_csv(index=False).__len__()

print("Approximate characters:", total_chars)
print("Approximate tokens:", total_chars // 4)

Number of rows: 234
Number of columns: 19
Approximate characters: 173028
Approximate tokens: 43257


In [9]:
# Convert the entire CSV into text for Ollama
dataset_context = df.to_csv(index=False)

print("Dataset characters:", len(dataset_context))
print("Approximate tokens:", len(dataset_context) // 4)

Dataset characters: 173028
Approximate tokens: 43257


In [20]:
import requests
import pandas as pd
import time

# ==========================================
# SETTINGS
# ==========================================

MODEL = "gemma3:1b"
OLLAMA_URL = "http://localhost:11434/api/generate"

results = []

# ==========================================
# RUN BATCH 1
# ==========================================

for index, row in batch_1.iterrows():

    question_id = row["Question ID"]

    # --------------------------------------
    # ENGLISH
    # --------------------------------------

    prompt_en = f"""
You are a dataset-only question answering system.

STRICT RULES:
1. Use ONLY information contained in the DATASET below.
2. Do NOT use your pretrained knowledge or outside information.
3. Do NOT guess.
4. If the answer cannot be found in the dataset, respond exactly: NOT FOUND.
5. Give only the answer.

DATASET:
{batch1_context}

QUESTION:
{row["English Question"]}

ANSWER:
"""

    response_en = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt_en,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    answer_en = response_en.json()["response"].strip()

    results.append({
        "Question ID": question_id,
        "Language": "English",
        "Question": row["English Question"],
        "LLM Answer": answer_en,
        "Dataset Answer": row["Gold Answer (EN)"],
        "Manual Verdict": "",
        "Error Type": ""
    })

    # --------------------------------------
    # NEPALI
    # --------------------------------------

    prompt_ne = f"""
तपाईं dataset मात्र प्रयोग गरेर प्रश्नको उत्तर दिने प्रणाली हुनुहुन्छ।

कडा नियमहरू:
1. तल दिइएको DATASET मा भएको जानकारी मात्र प्रयोग गर्नुहोस्।
2. आफ्नो pretrained knowledge वा बाहिरी जानकारी प्रयोग नगर्नुहोस्।
3. अनुमान नगर्नुहोस्।
4. DATASET मा उत्तर भेटिएन भने ठ्याक्कै NOT FOUND भन्नुहोस्।
5. सिधा उत्तर मात्र दिनुहोस्।

DATASET:
{batch1_context}

QUESTION:
{row["Nepali Question"]}

ANSWER:
"""

    response_ne = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt_ne,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    answer_ne = response_ne.json()["response"].strip()

    results.append({
        "Question ID": question_id,
        "Language": "Nepali",
        "Question": row["Nepali Question"],
        "LLM Answer": answer_ne,
        "Dataset Answer": row["Gold Answer (NE)"],
        "Manual Verdict": "",
        "Error Type": ""
    })

    print(f"Completed: {question_id}")

# ==========================================
# SAVE RESULTS
# ==========================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "Batch_1_Gemma3_1B_Results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n================================")
print("BATCH 1 COMPLETE")
print("Total responses:", len(results_df))
print("Saved as: Batch_1_Gemma3_1B_Results.csv")
print("================================")

Completed: GEO001
Completed: GEO002
Completed: GEO003
Completed: GEO004
Completed: GEO005
Completed: TOU001
Completed: TOU002
Completed: TOU003
Completed: TOU004
Completed: GEO006
Completed: TOU005
Completed: GEO007
Completed: TOU006
Completed: GEO008
Completed: GEO009
Completed: GEO010
Completed: GEO011
Completed: GEO012
Completed: GEO013
Completed: GEO014
Completed: GEO015
Completed: GEO016
Completed: GEO017
Completed: GEO018
Completed: GEO019
Completed: GEO020
Completed: GEO021
Completed: GEO022
Completed: GEO023
Completed: GEO024
Completed: GEO026
Completed: GEO027
Completed: GEO028
Completed: GEO029
Completed: GEO030
Completed: GEO031
Completed: TOU007
Completed: TOU008
Completed: TOU009
Completed: TOU010
Completed: TOU011
Completed: TOU012
Completed: TOU013
Completed: TOU014
Completed: TOU015
Completed: TOU016
Completed: TOU017
Completed: TOU018
Completed: TOU019
Completed: TOU020
Completed: TOU021
Completed: TOU022
Completed: TOU023
Completed: TOU024
Completed: TOU025
Completed:

In [2]:
import requests
import pandas as pd
import time

# ==========================================
# SETTINGS
# ==========================================

MODEL = "gemma3:1b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# Entire dataset
full_context = df.to_csv(index=False)

print("Dataset characters:", len(full_context))
print("Approximate tokens:", len(full_context) // 4)

results = []

# ==========================================
# RUN ALL 234 QUESTIONS × 2 LANGUAGES
# ==========================================

for index, row in df.iterrows():

    question_id = row["Question ID"]

    # --------------------------------------
    # ENGLISH
    # --------------------------------------

    prompt_en = f"""
You are a dataset-only question answering system.

STRICT RULES:
1. Use ONLY information contained in the DATASET below.
2. Do NOT use your pretrained knowledge or outside information.
3. Do NOT guess.
4. If the answer cannot be found in the dataset, respond exactly: NOT FOUND.
5. Give only the answer.

DATASET:
{full_context}

QUESTION:
{row["English Question"]}

ANSWER:
"""

    try:
        response_en = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt_en,
                "stream": False,
                "options": {
                    "temperature": 0
                }
            },
            timeout=600
        )

        if response_en.status_code == 200:
            answer_en = response_en.json()["response"].strip()
        else:
            answer_en = "REQUEST ERROR: " + response_en.text

    except Exception as e:
        answer_en = "REQUEST ERROR: " + str(e)

    results.append({
        "Question ID": question_id,
        "Language": "English",
        "Question": row["English Question"],
        "LLM Answer": answer_en,
        "Dataset Answer": row["Gold Answer (EN)"],
        "Manual Verdict": "",
        "Error Type": ""
    })

    print(f"{question_id} | English | Done")

    # --------------------------------------
    # NEPALI
    # --------------------------------------

    prompt_ne = f"""
तपाईं dataset मात्र प्रयोग गरेर प्रश्नको उत्तर दिने प्रणाली हुनुहुन्छ।

कडा नियमहरू:
1. तल दिइएको DATASET मा भएको जानकारी मात्र प्रयोग गर्नुहोस्।
2. आफ्नो pretrained knowledge वा बाहिरी जानकारी प्रयोग नगर्नुहोस्।
3. अनुमान नगर्नुहोस्।
4. DATASET मा उत्तर भेटिएन भने ठ्याक्कै NOT FOUND भन्नुहोस्।
5. सिधा उत्तर मात्र दिनुहोस्।

DATASET:
{full_context}

QUESTION:
{row["Nepali Question"]}

ANSWER:
"""

    try:
        response_ne = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "prompt": prompt_ne,
                "stream": False,
                "options": {
                    "temperature": 0
                }
            },
            timeout=600
        )

        if response_ne.status_code == 200:
            answer_ne = response_ne.json()["response"].strip()
        else:
            answer_ne = "REQUEST ERROR: " + response_ne.text

    except Exception as e:
        answer_ne = "REQUEST ERROR: " + str(e)

    results.append({
        "Question ID": question_id,
        "Language": "Nepali",
        "Question": row["Nepali Question"],
        "LLM Answer": answer_ne,
        "Dataset Answer": row["Gold Answer (NE)"],
        "Manual Verdict": "",
        "Error Type": ""
    })

    print(f"{question_id} | Nepali | Done")

    # Small pause to avoid hammering Ollama
    time.sleep(0.2)

# ==========================================
# SAVE RESULTS
# ==========================================

results_df = pd.DataFrame(results)

output_file = "Full_Dataset_Gemma3_1B_Results.csv"

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\n========================================")
print("EXPERIMENT COMPLETE")
print("Total responses:", len(results_df))
print("Expected responses:", len(df) * 2)
print("Saved:", output_file)
print("========================================")

NameError: name 'df' is not defined

In [2]:
import pandas as pd

file_path = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research\Hexagon-Dataset - Constitution&Law.csv"

df = pd.read_csv(file_path)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

df.head()

Number of rows: 917
Number of columns: 18


,Question ID,Domain,Subdomain,Question Type,Difficulty,English Question,Nepali Question,Gold Answer (EN),Gold Answer (NE),Explanation,Source ID,Source Title,Source URL,Source Type,Done,Reviewed By,Review Status,Notes
0,CL001,Constitution & Law,Preliminary,Fact Retrieval,Easy,What is the fundamental law of Nepal?,नेपालको मौलिक कानून के हो?,The Constitution of Nepal,नेपालको संविधान,Article 1 declares the Constitution as the fun...,Art. 1,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN
1,CL002,Constitution & Law,Preliminary,Fact Retrieval,Easy,"According to the Constitution, in whom is the ...",संविधानअनुसार नेपालको सार्वभौमसत्ता तथा राज्यस...,The Nepalese people,नेपाली जनता,Article 2 vests sovereignty and state authorit...,Art. 2,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN
2,CL003,Constitution & Law,Preliminary,Definition,Medium,How does the Constitution define the nation of...,संविधानले नेपाल राष्ट्रलाई कसरी परिभाषित गरेको छ?,"All the Nepalese people with multiethnic, mult...","बहुजातीय, बहुभाषिक, बहुधार्मिक, बहुसांस्कृतिक ...",Article 3 defines the nation of Nepal.,Art. 3,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN
3,CL004,Constitution & Law,Preliminary,Constitutional Provision,Easy,Which Article declares the Constitution to be ...,नेपालको संविधानलाई मौलिक कानून घोषणा गर्ने धार...,Article 1,धारा १,Article 1 establishes the Constitution as the ...,Art. 1,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN
4,CL005,Constitution & Law,Preliminary,Comparison,Medium,"According to the Constitution, who exercises s...","संविधानअनुसार राज्यसत्ता कसमा निहित छ, र त्यो ...",State authority is vested in the Nepalese peop...,राज्यसत्ता नेपाली जनतामा निहित हुन्छ र संविधान...,This combines both clauses of Article 2.,Art. 2,Constitution of Nepal (2015),https://lawcommission.gov.np/en/?cat=152,Constitution,Aayesha,NaN,NaN,NaN


In [4]:
full_context = df.to_csv(index=False)

print("Dataset characters:", len(full_context))
print("Approximate tokens:", len(full_context) // 4)

Dataset characters: 481168
Approximate tokens: 120292


In [1]:
import requests

response = requests.post(
    "http://localhost:11434/api/show",
    json={"name": "gemma3:1b"}
)

print(response.status_code)
print(response.json())

200
{'license': 'Gemma Terms of Use \n\nLast modified: February 21, 2024\n\nBy using, reproducing, modifying, distributing, performing or displaying any portion or element of Gemma, Model Derivatives including via any Hosted Service, (each as defined below) (collectively, the "Gemma Services") or otherwise accepting the terms of this Agreement, you agree to be bound by this Agreement.\n\nSection 1: DEFINITIONS\n1.1 Definitions\n(a) "Agreement" or "Gemma Terms of Use" means these terms and conditions that govern the use, reproduction, Distribution or modification of the Gemma Services and any terms and conditions incorporated by reference.\n\n(b) "Distribution" or "Distribute" means any transmission, publication, or other sharing of Gemma or Model Derivatives to a third party, including by providing or making Gemma or its functionality available as a hosted service via API, web access, or any other electronic or remote means ("Hosted Service").\n\n(c) "Gemma" means the set of machine le